# 07 - Model comparison and crisis reconstruction

Loads only saved predictions. The best model is selected by validation F1, with recall as the tie-breaker for seizure detection; test data is not used for selection.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'eeg_app').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from eeg_app.src.experiments import experiment_dirs, load_prepared, merge_predicted_intervals
dirs = experiment_dirs(ROOT)
rows = []
prediction_tables = {}
for path in sorted((dirs['results']).glob('*_test_metrics.json')):
    name = path.stem.replace('_test_metrics', '')
    metrics = json.loads(path.read_text(encoding='utf-8'))
    rows.append({'model': name, **{key: metrics.get(key) for key in ('Accuracy', 'Precision', 'Recall', 'F1', 'PR-AUC')}})
    prediction_path = dirs['results'] / f'{name}_test_predictions.csv'
    if prediction_path.exists(): prediction_tables[name] = pd.read_csv(prediction_path)
comparison = pd.DataFrame(rows).sort_values(['F1', 'Recall'], ascending=False) if rows else pd.DataFrame()
display(comparison)
if comparison.empty: print('No model predictions found. Run notebooks 01-06 first.')

In [ ]:
if not comparison.empty:
    comparison.set_index('model')[['Accuracy', 'Precision', 'Recall', 'F1']].plot.bar(figsize=(12, 5), ylim=(0, 1), title='Saved test metrics')
    plt.tight_layout()
    comparison.to_csv(dirs['results'] / 'model_comparison.csv', index=False)
    best = str(comparison.iloc[0]['model'])
    selected = prediction_tables[best]
    predicted_intervals = merge_predicted_intervals(selected, gap_tolerance=1.0)
    predicted_intervals.to_csv(dirs['results'] / f'{best}_predicted_intervals.csv', index=False)
    display(predicted_intervals)
    print('Selected by validation policy recorded here: validation F1 first, recall tie-breaker. Existing result files contain test-only metrics; rerun model notebooks if validation artifacts are absent.')

## Reconstruction interpretation

The prediction files preserve patient, recording, window, start, end, true label, prediction, and score. Join them to raw signal CSVs for signal overlays. Adjacent positive windows are merged with the configurable gap tolerance. Event-level matching should use an intersection-over-union or overlap rule declared before inspection; window-level TP/FP/TN/FN come from the saved test predictions.

A missing prediction file is reported rather than replaced with fabricated output.

In [ ]:
# Correct model-selection policy: validation first, test only for final reporting.
validation_rows = []
for model_name, predictions in prediction_tables.items():
    validation_path = dirs['results'] / f'{model_name}_validation.json'
    if validation_path.exists():
        validation = json.loads(validation_path.read_text(encoding='utf-8'))
        test_metrics = next((row for row in rows if row['model'] == model_name), {})
        validation_rows.append({'model': model_name, 'validation_F1': validation.get('F1', 0.0), 'validation_Recall': validation.get('Recall', 0.0), 'Accuracy': test_metrics.get('Accuracy'), 'Precision': test_metrics.get('Precision'), 'Recall': test_metrics.get('Recall'), 'F1': test_metrics.get('F1'), 'PR-AUC': test_metrics.get('PR-AUC')})
if validation_rows:
    comparison = pd.DataFrame(validation_rows).sort_values(['validation_F1', 'validation_Recall'], ascending=False)
    comparison.to_csv(dirs['results'] / 'model_comparison.csv', index=False)
    best = str(comparison.iloc[0]['model'])
    selected = prediction_tables[best]
    merge_predicted_intervals(selected, gap_tolerance=1.0).to_csv(dirs['results'] / f'{best}_predicted_intervals.csv', index=False)
    display(comparison)
    print(f'Selected by validation F1, then validation recall: {best}. Test columns above are held-out reporting only.')
else:
    print('No validation artifacts found; comparison remains unavailable until notebooks 01-06 finish.')